# Morphological Profiling

Evaluate segmentation models on the **BBBC021 dataset** — a high-throughput fluorescence microscopy screen of MCF-7 breast cancer cells. The pipeline benchmarks models on their ability to preserve morphological signal, measured by how well unsupervised clustering separates Mechanism of Action (MoA) classes.

**Dataset:** 13,200 fields, 113 compounds at 8 concentrations, 3 channels (DAPI, Actin, Tubulin). 38 compounds (103 compound-concentration pairs) annotated with 12 MoA classes + DMSO control.

**Pipeline:**
1. Preprocess metadata
2. Segmentation (DAPI channel)
3. Feature extraction (~106 features per cell)
4. Aggregation & normalization
5. Unsupervised clustering & evaluation

In [ ]:
import sys
from pathlib import Path

# Setup paths
SRC_DIR = Path.cwd().parent / 'src'
MORPH_DIR = SRC_DIR / 'morphology_profiling'
if str(MORPH_DIR) not in sys.path:
    sys.path.insert(0, str(MORPH_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config
config.ensure_dirs()

print(f'Project root: {config.PROJECT_ROOT}')
print(f'Data root: {config.DATA_ROOT}')
print(f'Results dir: {config.RESULTS_DIR}')

## Download Data

Download metadata and plate images from [BBBC021](https://bbbc.broadinstitute.org/BBBC021):

In [ ]:
# Download metadata only (~4 MB) - run this first
# !python {MORPH_DIR / 'download.py'} --metadata_only

# Download a single plate for testing (~800 MB)
# !python {MORPH_DIR / 'download.py'} --plate Week1_22123

# Download all 55 plates (~44 GB)
# !python {MORPH_DIR / 'download.py'} --all

# Check what data is available
if config.METADATA_DIR.exists():
    csvs = list(config.METADATA_DIR.glob('*.csv'))
    print(f'Metadata files: {[f.name for f in csvs]}')
if config.IMAGES_DIR.exists():
    plates = [d.name for d in config.IMAGES_DIR.iterdir() if d.is_dir()]
    print(f'Downloaded plates: {len(plates)} ({plates[:5]}...)')
else:
    print('No image data found. Run download commands above.')

## Step 1: Preprocess Metadata

Parse compound/concentration/MoA mappings and build a unified image table.

In [ ]:
from preprocess import run_preprocessing

df, summary_df = run_preprocessing()

print(f'\nImage table: {len(df):,} rows')
print(f'Treatment summary: {len(summary_df)} treatments')
print(f'\nMoA classes: {df[df["moa"] != "unknown"]["moa"].nunique()}')
print(f'DMSO fields: {df["is_dmso"].sum()}')

## Step 2: Segmentation

Run instance segmentation on the DAPI channel for all fields.

**Models:** cellpose4, cellpose3, microatlas, microsam, cellsam

In [ ]:
from segment import load_model, segment_plate

# Select model
MODEL_NAME = 'microatlas'  # Options: 'cellpose4', 'cellpose3', 'microatlas', 'microsam', 'cellsam'

# Get available plates
plates = [d.name for d in sorted(config.IMAGES_DIR.iterdir())
          if d.is_dir() and (d / config.NUCLEUS_CHANNEL).exists()]
print(f'Available plates: {len(plates)}')
print(f'Model: {MODEL_NAME}')

# Load model
model = load_model(MODEL_NAME, use_gpu=True)
config.ensure_model_dirs(MODEL_NAME)

# Segment all plates
for plate in plates:
    print(f'\n--- Plate: {plate} ---')
    segment_plate(plate, MODEL_NAME, model)

## Step 3: Feature Extraction

Extract ~106 morphological features per cell across 6 categories:

| Category | Description | Features |
|---|---|---|
| AreaShape | regionprops shape descriptors | 13 |
| Intensity | batched ndimage stats + percentiles x 3ch | 38 |
| Texture | Haralick GLCM on DAPI + Actin, 1 scale | 26 |
| Granularity | multi-scale opening on DAPI, 5 scales | 5 |
| RadialDistribution | binned radial intensity x 3ch | 12 |
| Correlation | Pearson + Manders inter-channel | 12 |

In [ ]:
from feature_extraction import extract_features_for_plate

# Extract features for all plates
for plate in plates:
    print(f'\n--- Extracting features: {plate} ---')
    extract_features_for_plate(plate, MODEL_NAME)

# Check output
feat_dir = config.features_dir(MODEL_NAME)
feat_files = list(feat_dir.glob('features_*.csv'))
print(f'\nFeature files generated: {len(feat_files)}')

## Step 4: Aggregation & Normalization

Aggregate single-cell features to field-level (median + MAD), robust z-score against DMSO controls, remove low-variance (< 0.01) and highly correlated (> 0.95) features.

In [ ]:
from feature_aggregation import load_all_features, aggregate_and_normalize

# Load all single-cell features
df_features = load_all_features(model_name=MODEL_NAME)
print(f'Single-cell features: {len(df_features):,} cells, {len(df_features.columns)} columns')

# Aggregate to field-level, normalize, and select features
df_field = aggregate_and_normalize(df_features, model_name=MODEL_NAME)
print(f'\nField-level profiles: {len(df_field)} fields')
print(f'Features after selection: {len([c for c in df_field.columns if c not in {"plate","field","compound","concentration","moa","is_dmso","well","n_cells"}])}')

## Step 5: Unsupervised Clustering & Evaluation

Field-level profiles are reduced via PCA (50 dims) then UMAP (5D, cosine distance), aggregated to treatment-level centroids, and clustered via HDBSCAN. Quality is measured by Hungarian-matched Accuracy — the proportion of valid treatments correctly assigned to their MoA class.

In [ ]:
from biomarker.unsupervised import run_full_unsupervised_pipeline

# Run the full unsupervised clustering pipeline
results = run_full_unsupervised_pipeline(df_field, model_name=MODEL_NAME)

if results is not None:
    accuracy, nmi, cluster_df = results
    print(f'\n{"="*60}')
    print(f'Results for {MODEL_NAME}:')
    print(f'  Hungarian Accuracy: {accuracy:.4f}')
    print(f'  NMI: {nmi:.4f}')
    print(f'{"="*60}')